In [0]:
%sql
USE rishikesh_db;

In [0]:
%sql

-- 2. Check for NULLs in critical columns (Should return 0)
-- This ensures every transaction has an ID and a Customer

SELECT 'Null Check' AS Test_Name, COUNT(*) AS Failed_Records
FROM silver_sales_enriched
WHERE order_id IS NULL OR customer_name IS NULL;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6804946366301028>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "\n-- 2. Check for NULLs in critical columns (Should return 0)\n-- This ensures every transaction has an ID and a Customer\n\nSELECT 'Null Check' AS Test_Name, COUNT(*) AS Failed_Records\nFROM silver_sales_enriched\nWHERE order_id IS NULL OR customer_name IS NULL;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, 

In [0]:
%sql
-- 3. Check for Duplicate Orders (Should return 0)
-- Even though we deduplicated in Silver, this is a final "safety net"

select 'Duplicate Check' as test_Name, (Count(*)) - Count(Distinct (order_id)) as Failed_Records
from silver_sales_enriched;

test_Name,Failed_Records
Duplicate Check,0


In [0]:
%sql
-- 4. Check for unexpected Row Count drops
-- If we have fewer than 3,500 rows, something went wrong in the Bronze-to-Silver load




Select 'Volumme Check' as Test_Name, 
        Case when count(*)< 3500 Then 'FAIL: Low Volume' Else 'PASS' END as Status
From silver_sales_enriched;


Test_Name,Status
Volumme Check,PASS


### 🚨 Automated Alert Logic
In a production workflow, if any of the "Failed_Records" above are greater than 0, we would trigger an **Assertion Error** to stop the pipeline.

### 🚑 Data Quarantine Logic
Records that fail our quality checks are moved to a **Quarantine Table** for manual inspection, ensuring the Gold layer remains 100% clean.

In [0]:
%sql
-- 1. Drop the existing quarantine table to clear the mismatch
DROP TABLE IF EXISTS rishikesh_db.quarantine_sales;

-- 2. Recreate it by cloning the LATEST schema from your Silver table
CREATE TABLE rishikesh_db.quarantine_sales
USING DELTA
AS SELECT * FROM rishikesh_db.silver_sales_enriched WHERE 1=0;

-- 3. Now the INSERT will work perfectly because the columns match
INSERT INTO rishikesh_db.quarantine_sales
SELECT * FROM rishikesh_db.silver_sales_enriched
WHERE order_id IS NULL OR customer_name IS NULL;

-- 4. Verify it's ready for the 4,000-row pipeline
SELECT * FROM rishikesh_db.quarantine_sales;

order_id,customer_id,customer_name,sale_amount,silver_load_time,governance_check


### 🛑 Workflow Stop (Kill Switch)
If the number of quarantined rows is too high, we stop the entire pipeline to prevent bad data from reaching our stakeholders.

In [0]:
# Check how many rows failed the quality gate
failed_count = spark.sql("SELECT COUNT(*) FROM rishikesh_db.quarantine_sales").collect()[0][0]

# If more than 5% of our 4,000 rows are bad, we stop the pipeline!
if failed_count > 200:
    raise Exception(f"CRITICAL ERROR: {failed_count} rows quarantined. Data quality threshold failed!")
else:
    print(f"PASS: {failed_count} rows quarantined. Proceeding to Gold Layer.")

PASS: 0 rows quarantined. Proceeding to Gold Layer.
